## Generate MCR (Multiple Condition Run) for PTO Damping
Create WEC-Sim MCR file from MHKIT Pac-Wave example. Outputs best best clustering configuration based on energy flux for representative JONSWAP sea states.

USER INPUTS: Change these for your specific application.

In [8]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from scipy.io import savemat
from mhkit.wave import resource
from mhkit.wave.io import ndbc

# USER INPUTS
PROJECT_ROOT = Path.home() / "WEC-Sim_Applications" / "OSWEC_Optimization_Damping"
buoy_number = "46050"
years = ["2020", "2021", "2022", "2023", "2024", "2025"]
clusters = [4, 8, 16, 32, 64]
water_depth = 399.0
random_state = 1

# PTO damping values for WEC-Sim MCR sweep
damping_values = np.array([1e4, 1e5, 1e6, 1e7, 1e8], dtype=float)

# PROJECT FOLDER SETUP
os.chdir(PROJECT_ROOT)
output_folder = PROJECT_ROOT / "wave_conditions"
output_folder.mkdir(parents=True, exist_ok=True)

Download spectral wave energy density data from NDBC and clean data.

In [9]:
parameter = "swden"
ndbc_available_data = ndbc.available_data(parameter, buoy_number)

year_pattern = "|".join(years)
ndbc_available_data = ndbc_available_data[
    ndbc_available_data["filename"].astype(str).str.contains(year_pattern)
]

filenames = ndbc_available_data["filename"]
ndbc_requested_data = ndbc.request_data(parameter, filenames)

# CLEAN NDBC DATA
ndbc_data = {}

for year in ndbc_requested_data:
    data_raw = ndbc_requested_data[year].copy()

    data_raw["date"] = pd.to_datetime({
        "year": data_raw["#YY"],
        "month": data_raw["MM"],
        "day": data_raw["DD"],
        "hour": data_raw["hh"],
        "minute": data_raw["mm"],
    }, errors="coerce")

    data_raw = data_raw.set_index("date").drop(
        columns=["#YY", "MM", "DD", "hh", "mm"]
    )

    data_raw.columns = [
        float(str(col).replace(".", "0.")) if str(col).startswith(".") else float(col)
        for col in data_raw.columns
    ]

    data_raw = (
        data_raw
        .apply(pd.to_numeric, errors="coerce")
        .replace([999.0, 99.0], np.nan)
        .dropna()
        .sort_index()
    )

    ndbc_data[str(year)] = data_raw


Calculate wave parameters.

In [10]:
# CALCULATE WAVE PARAMETERS
print("Calculating wave parameters...")

Hm0_list = []
Te_list = []
J_list = []

for year in ndbc_data:
    spectrum = ndbc_data[year].T

    Hm0_list.append(resource.significant_wave_height(spectrum))
    Te_list.append(resource.energy_period(spectrum))
    J_list.append(resource.energy_flux(spectrum, h=water_depth))

Te = pd.concat(Te_list, axis=0)
Hm0 = pd.concat(Hm0_list, axis=0)
J = pd.concat(J_list, axis=0)

data = pd.concat([Hm0, Te, J], axis=1)
data.columns = ["Hm0", "Te", "J"]

# CLEAN DATA
data_clean = data.copy()
data_clean = data_clean[data_clean.Hm0 < 20]

sigma = data_clean.J.std()
data_clean = data_clean[data_clean.J > (data_clean.J.mean() - 0.9 * sigma)]

total_energy_flux = data_clean.J.sum()

print(f"\nTotal energy flux from all sea states: {total_energy_flux:.2f}")

Calculating wave parameters...

Total energy flux from all sea states: 2478398192.00


Clustering by means of Gaussian mixture model.

In [11]:
# CLUSTERING
X = np.vstack((data_clean.Te.values, data_clean.Hm0.values)).T

results = {}
ratios = {}

# Frequency vector from NDBC data
f = ndbc_data[list(ndbc_data.keys())[0]].columns.values

print("\nCalculating clusters and energy flux ratios:")
print("--------------------------------------------")

for N in clusters:
    print(f"Processing cluster size: {N}...")

    gmm = GaussianMixture(
        n_components=N,
        random_state=random_state
    ).fit(X)

    result = pd.DataFrame(gmm.means_, columns=["Te", "Hm0"])

    result["weights"] = gmm.weights_ / gmm.weights_.sum()
    result["Tp"] = result["Te"] / 0.858

    result = result.sort_values(["Hm0", "Te"]).reset_index(drop=True)

    result.insert(0, "condition_id", np.arange(1, len(result) + 1))

    result["probability"] = result["weights"]
# Estimate energy flux for representative JONSWAP sea states
    J_cluster = []

    for i in range(len(result)):
        spectrum = resource.jonswap_spectrum(
            f,
            result.loc[i, "Tp"],
            result.loc[i, "Hm0"]
        )

        J_cluster.append(resource.energy_flux(spectrum, h=water_depth).item())

    result["J"] = J_cluster

    results[N] = result

    J_hr = result["J"] * len(data_clean)
    total_weighted_J = (J_hr * result["weights"]).sum()
    normalized_weighted_J = total_weighted_J / total_energy_flux

    ratios[N] = np.round(normalized_weighted_J, 4)

    print(f"  Cluster {N}:")
    print(f"    Total weighted J: {total_weighted_J:.2f}")
    print(f"    Ratio: {ratios[N]:.4f}")

print("\n--------------------------------------------")
print("All Cluster Ratios:")

for N, ratio in sorted(ratios.items()):
    print(f"  Cluster {N}: Ratio = {ratio:.4f}")
    



Calculating clusters and energy flux ratios:
--------------------------------------------
Processing cluster size: 4...
  Cluster 4:
    Total weighted J: 2166183043.28
    Ratio: 0.8740
Processing cluster size: 8...
  Cluster 8:
    Total weighted J: 2225649273.20
    Ratio: 0.8980
Processing cluster size: 16...
  Cluster 16:
    Total weighted J: 2305156512.74
    Ratio: 0.9301
Processing cluster size: 32...
  Cluster 32:
    Total weighted J: 2349453990.88
    Ratio: 0.9480
Processing cluster size: 64...
  Cluster 64:
    Total weighted J: 2377366190.97
    Ratio: 0.9592

--------------------------------------------
All Cluster Ratios:
  Cluster 4: Ratio = 0.8740
  Cluster 8: Ratio = 0.8980
  Cluster 16: Ratio = 0.9301
  Cluster 32: Ratio = 0.9480
  Cluster 64: Ratio = 0.9592


Select best cluster based on energy flux for representative JONSWAP sea states.

In [12]:
# SELECT BEST CLUSTER
best_cluster = min(ratios, key=lambda x: abs(ratios[x] - 1.0))
best_ratio = ratios[best_cluster]

print(f"\nSelected best cluster: {best_cluster} condition(s)")
print(f"Ratio = {best_ratio:.4f}")
print(f"Difference from 1.0 = {abs(best_ratio - 1.0):.4f}")

# FINAL REPRESENTATIVE WAVE CONDITIONS
result = results[best_cluster].copy()

# Normalize probabilities
result["probability"] = result["probability"] / result["probability"].sum()

# BUILD FINAL WEC-SIM MCR FILE DIRECTLY
mcr_cases = []
case_number = []
wave_id = []
damping_id = []
probability = []
Hm0_values = []
Tp_values = []
Te_values = []
damping_case_values = []

case_counter = 0

for j, damping in enumerate(damping_values, start=1):
    for _, row in result.iterrows():
        case_counter += 1

        mcr_cases.append([
            float(row["Hm0"]),
            float(row["Tp"]),
            float(damping),
        ])

        case_number.append(case_counter)
        wave_id.append(int(row["condition_id"]))
        damping_id.append(j)
        probability.append(float(row["probability"]))
        Hm0_values.append(float(row["Hm0"]))
        Tp_values.append(float(row["Tp"]))
        Te_values.append(float(row["Te"]))
        damping_case_values.append(float(damping))

mcr_cases = np.array(mcr_cases, dtype=float)


Selected best cluster: 64 condition(s)
Ratio = 0.9592
Difference from 1.0 = 0.0408


Output file set up. Change these for your specific set up. Currently prints to 
``WEC-Sim_Applications/OSWEC_Optimization_Damping\wave_conditions``.

In [13]:
# OUTPUT FILES
years_tag = "_".join(years)

output_base_name = (
    f"wave_conditions_buoy_{buoy_number}_{years_tag}_"
    f"{best_cluster}_clusters_damping_mcr"
)

output_mat_file = output_folder / f"{output_base_name}.mat"
output_csv_file = output_folder / f"{output_base_name}.csv"

# WEC-Sim MCR structure
mcr = {
    "header": np.array(
        [["waves.height", "waves.period", "pto(1).damping"]],
        dtype=object,
    ),

    "cases": mcr_cases,

    # Metadata for userDefinedFunctions.m
    "case_number": np.array(case_number, dtype=float),
    "wave_id": np.array(wave_id, dtype=float),
    "damping_id": np.array(damping_id, dtype=float),
    "probability": np.array(probability, dtype=float),
    "Hm0": np.array(Hm0_values, dtype=float),
    "Tp": np.array(Tp_values, dtype=float),
    "Te": np.array(Te_values, dtype=float),
    "damping": np.array(damping_case_values, dtype=float),
    "damping_values": damping_values,
}

wave_conditions = {
    "condition_id": result["condition_id"].to_numpy(dtype=float),
    "Hm0": result["Hm0"].to_numpy(dtype=float),
    "Te": result["Te"].to_numpy(dtype=float),
    "Tp": result["Tp"].to_numpy(dtype=float),
    "probability": result["probability"].to_numpy(dtype=float),
    "J": result["J"].to_numpy(dtype=float),
}

cluster_selection = {
    "cluster_sizes": np.array(list(ratios.keys()), dtype=float),
    "ratios": np.array(list(ratios.values()), dtype=float),
    "best_cluster": float(best_cluster),
    "best_ratio": float(best_ratio),
    "total_energy_flux": float(total_energy_flux),
}

savemat(str(output_mat_file), {
    "mcr": mcr,
    "wave_conditions": wave_conditions,
    "cluster_selection": cluster_selection,
})

# CSV copy of expanded MCR cases
mcr_csv = pd.DataFrame({
    "case_number": case_number,
    "wave_id": wave_id,
    "damping_id": damping_id,
    "Hm0": Hm0_values,
    "Tp": Tp_values,
    "Te": Te_values,
    "probability": probability,
    "pto_damping": damping_case_values,
})

mcr_csv.to_csv(output_csv_file, index=False)

print("\nComplete.")
print(f"Created WEC-Sim MCR MAT file: {output_mat_file}")
print(f"Created CSV file: {output_csv_file}")
print(f"Wave conditions: {len(result)}")
print(f"Damping values: {len(damping_values)}")
print(f"Total MCR cases: {len(mcr_cases)}")
print(f"Selected cluster: {best_cluster}")
print(f"Energy flux ratio: {best_ratio:.4f}")


Complete.
Created WEC-Sim MCR MAT file: C:\Users\rsaucie\WEC-Sim_Applications\OSWEC_Optimization_Damping\wave_conditions\wave_conditions_buoy_46050_2020_2021_2022_2023_2024_2025_64_clusters_damping_mcr.mat
Created CSV file: C:\Users\rsaucie\WEC-Sim_Applications\OSWEC_Optimization_Damping\wave_conditions\wave_conditions_buoy_46050_2020_2021_2022_2023_2024_2025_64_clusters_damping_mcr.csv
Wave conditions: 64
Damping values: 5
Total MCR cases: 320
Selected cluster: 64
Energy flux ratio: 0.9592
